In [ ]:
#| default_exp core

# FastCDP API
> Source and API details

In [ ]:
#| export
from fastcore.utils import *
import shutil, websockets, json, platform, asyncio, inspect, base64, httpx
from contextlib import asynccontextmanager

In [ ]:
#| export
if '__file__' not in globals():
    _root = Path.cwd()
    if not (Path.cwd()/'pyproject.toml').exists(): _root = _root.parent
    __file__ = _root/'fastcdp'/'core.py'

_path = Path(__file__).parent
_bp = (_path/'browser_protocol.json').read_json()
_jp = (_path/'js_protocol.json').read_json()
_cdp_domains = _bp['domains'] + _jp['domains']

In [ ]:
len(_cdp_domains), [d['domain'] for d in _cdp_domains[:5]]

(55, ['Accessibility', 'Animation', 'Audits', 'Autofill', 'BackgroundService'])

In [ ]:
#| export
def cdp_search(q:str):
    "Search CDP domains and commands by name or description"
    q = q.lower()
    res = []
    for d in _cdp_domains:
        for cmd in d.get('commands', []):
            desc = cmd.get('description', '')
            if q in cmd['name'].lower() or q in desc.lower():
                res.append(f"{d['domain']}.{cmd['name']}: {desc[:120]}")
        for evt in d.get('events', []):
            desc = evt.get('description', '')
            if q in evt['name'].lower() or q in desc.lower():
                res.append(f"  evt {d['domain']}.{evt['name']}: {desc[:120]}")
    return '\n'.join(res)

In [ ]:
cdp_search('target')[:100]

'Audits.checkFormsIssues: Runs the form issues check for the target page. Found issues are reported\nu'

In [ ]:
#| export
def _lower1(s): return s[0].lower() + s[1:]
def _upper1(s): return s[0].upper() + s[1:]

_chrome_paths = dict(
    Darwin=['Library/Application Support/Google/Chrome/DevToolsActivePort',
            'Library/Application Support/Chromium/DevToolsActivePort'],
    Linux=['.config/google-chrome/DevToolsActivePort',
           '.config/chromium/DevToolsActivePort'])

def cdp_conninfo(p=None, d=None):
    "Connection info from contents `p`, profile dir `d`, or the default Chrome profile"
    if d: p = (Path(d)/'DevToolsActivePort').read_text()
    if not p:
        paths = _chrome_paths.get(platform.system(), _chrome_paths['Linux'])
        f = first(Path.home()/p for p in paths if (Path.home()/p).exists())
        if not f: raise FileNotFoundError('No Chrome or Chromium DevToolsActivePort found')
        p = f.read_text()
    lines = p.strip().split('\n')
    return ''.join(lines[:2])

Chrome (146+) can expose CDP from your everyday browser: enable **Allow remote debugging** in `chrome://inspect`. Chrome then accepts WebSocket connections on the endpoint recorded in its profile's `DevToolsActivePort` file -- port on the first line, WebSocket path on the second -- and asks you to approve each newly connecting client. Only that WebSocket endpoint is served (the `/json/*` HTTP interface belongs to the debug-instance mechanism described later). `cdp_conninfo` reads the file, and `CDP.connect()` uses it by default.

On MacOS, the connection info is stored in `~/Library/Application Support/Google/Chrome/DevToolsActivePort` (or `~/Library/Application Support/Chromium/DevToolsActivePort` for Chromium).

In [ ]:
conninfo = cdp_conninfo()
conninfo

'9222/devtools/browser/6e04b056-236c-45c3-b0d5-597d0e45723e'

In [ ]:
#| export
class CDP:
    "Chrome DevTools Protocol connection with event support"
    def __init__(self, wsconn=None, debug=False):
        self._id,self._pending,self._events = 0,{},{}
        store_attr()

    @classmethod
    async def connect(cls, p=None, wsconn=None, debug=None):
        if wsconn is None: wsconn = cdp_conninfo()
        self = cls(wsconn, debug=debug)
        url = self.wsconn if self.wsconn.startswith('ws') else f'ws://127.0.0.1:{self.wsconn}'
        self.ws = await websockets.connect(url)
        self._reader = asyncio.create_task(self._read_loop())
        self._keep = asyncio.create_task(self._keepalive())
        return self

    async def _read_loop(self):
        try:
            async for raw in self.ws:
                msg = json.loads(raw)
                if 'id' in msg:
                    if (fut := self._pending.pop(msg['id'], None)): fut.set_result(msg)
                else: self._dispatch(msg)
        except websockets.ConnectionClosed: pass

    def _dispatch(self, msg):
        "Route an event frame to every queue subscribed to its method"
        if 'method' not in msg: return
        if self.debug: print(f"EVT: {msg['method']} sid={msg.get('sessionId','')[:8]}")
        for q in self._events.get(msg['method'], []): q.put_nowait(msg)

    async def _keepalive(self):
        while True:
            try: await self('Runtime.evaluate', expression='1')
            except: break
            await asyncio.sleep(30)

    async def _send(self, msg):
        "Send one command frame and return its reply frame; transports override this"
        self._id += 1
        msg['id'] = self._id
        fut = asyncio.get_event_loop().create_future()
        self._pending[self._id] = fut
        await self.ws.send(json.dumps(msg))
        return await fut

    async def __call__(self, method, sid=None, **params):
        msg = dict(method=method)
        if params: msg['params'] = params
        if sid: msg['sessionId'] = sid
        r = await self._send(msg)
        if 'error' in r: raise RuntimeError(f"{method}: {r['error']}")
        res = r.get('result', {})
        return first(res.values()) if len(res) == 1 else res

    async def close(self):
        self._keep.cancel()
        self._reader.cancel()
        if (t := getattr(self, '_dialog_task', None)): t.cancel()
        await self.ws.close()

    @property
    def is_open(self): return self.ws.state.name == 'OPEN'

    def __dir__(self): return super().__dir__() + [_lower1(o) for o in _domains.keys()]

    def __getattr__(self, domain):
        if domain.startswith('_'): raise AttributeError()
        return CDPDomain(self, _upper1(domain))

The transport is pluggable. `_send` (command frame in, reply frame out) and `_dispatch` (route an event frame to subscribed queues) are the only two methods that touch the websocket protocol flow, so an alternative transport subclasses `CDP`, overrides `_send` and the connection lifecycle, and feeds incoming events to `_dispatch`. Everything else (domain proxies, helpers, event buffers, `Page`) is inherited unchanged. `solvecdp` does exactly this, relaying frames through a solveit server to a Chrome extension.

In [ ]:
# await cdp.close()

In [ ]:
#| export
@patch(cls_method=True)
async def remote(cls:CDP, port=9223, debug=None):
    "Connect via Chrome remote debugging HTTP endpoint"
    async with httpx.AsyncClient() as client:
        url = (await client.get(f'http://localhost:{port}/json/version')).json()['webSocketDebuggerUrl']
    return await cls.connect(wsconn=url, debug=debug)

`remote` targets the other mechanism: a separate "debug Chrome" started with a remote debugging port. `fastcdp-setup` creates a "CDP Chrome" launcher for one on `remote`'s default port 9223 (not 9222, which a main browser with built-in debugging enabled already holds). Since Chrome 136 the debugging switches are ignored for your everyday profile -- a debug instance must point `--user-data-dir` at a non-standard directory, so that scripts can't reach your real profile's (differently-encrypted) cookies. To start one by hand on macOS:

```sh
'/Applications/Google Chrome.app/Contents/MacOS/Google Chrome' \
  --remote-debugging-port=9223 --user-data-dir=$HOME/.cache/fastcdp/cdp-chrome
```

That instance serves the classic HTTP metadata endpoints, and `remote` reads `webSocketDebuggerUrl` from `/json/version` to connect -- no approval prompts, since the profile is disposable.

## Launching Chrome

The third connection option: start the user's installed Chrome ourselves, CDP-ready -- no manual setup, no approval popups. `launch` runs it on a separate profile directory (Chrome requires a non-default one for debugging) with an ephemeral debug port, and `quit` shuts it down again. One launched instance per profile dir: a second `launch` on the same dir would just signal the running instance and exit.

We launch `headless=True` here so running this notebook's tests doesn't pop up a browser window; drop it when you want to watch.

In [ ]:
#| export
_chrome_bins = dict(Darwin=['/Applications/Google Chrome.app/Contents/MacOS/Google Chrome',
    '/Applications/Chromium.app/Contents/MacOS/Chromium'],
    Linux=['google-chrome', 'chromium', 'chromium-browser'])

def chrome_bin():
    "Path of the installed Chrome/Chromium binary (`$FASTCDP_CHROME` overrides)"
    if (p := os.environ.get('FASTCDP_CHROME')): return p
    bins = _chrome_bins.get(platform.system(), _chrome_bins['Linux'])
    b = first(b for b in bins if Path(b).exists() or shutil.which(b))
    if not b: raise FileNotFoundError('No Chrome or Chromium found; set FASTCDP_CHROME')
    return b

In [ ]:
#| export
@patch(cls_method=True)
async def launch(cls:CDP, user_data_dir=None, headless=False, debug=None, timeout=10):
    "Launch the installed Chrome CDP-ready on its own profile dir, and connect to it"
    d = Path(user_data_dir) if user_data_dir else Path.home()/'.cache'/'fastcdp'/'profile'
    d.mkdir(parents=True, exist_ok=True)
    (d/'DevToolsActivePort').unlink(missing_ok=True)
    args = [f'--user-data-dir={d}', '--remote-debugging-port=0', '--no-first-run', '--no-default-browser-check']
    if headless: args.append('--headless=new')
    proc = await asyncio.create_subprocess_exec(chrome_bin(), *args,
        stdout=asyncio.subprocess.DEVNULL, stderr=asyncio.subprocess.DEVNULL)
    deadline = asyncio.get_event_loop().time() + timeout
    while not (d/'DevToolsActivePort').exists():
        if proc.returncode is not None: raise RuntimeError('Chrome exited at launch (already running on this profile dir?)')
        if asyncio.get_event_loop().time() > deadline: raise TimeoutError('Timed out waiting for Chrome debug endpoint')
        await asyncio.sleep(0.05)
    ci = cdp_conninfo(d=d)
    self = await cls.connect(wsconn=ci, debug=debug)
    self.proc,self.port = proc, int(ci.split('/', 1)[0])
    return self

In [ ]:
#| export
@patch
async def quit(self:CDP):
    "Quit the browser and close the connection"
    try: await asyncio.wait_for(self('Browser.close'), 2)
    except (TimeoutError, websockets.ConnectionClosed): pass
    await self.close()
    if (proc := getattr(self, 'proc', None)) is not None: await proc.wait()

In [ ]:
lcdp = await CDP.launch(headless=False)
lcdp.port

63744

In [ ]:
cdp = await CDP.remote(port=lcdp.port)

In [ ]:
#| export
@patch(as_prop=True)
async def pages(self:CDP):
    return [t for t in (await self('Target.getTargets')) if t['type'] == 'page']

In [ ]:
await cdp('Target.createTarget', url='https://example.com')

'9D4B109315038DA34934A12132DAAD3A'

In [ ]:
ps = await cdp.pages
pg = first(p for p in ps if 'example.com' in p['url'])
pg['title']

'Example Domain'

In [ ]:
#| export
_domains = {d['domain']: d for d in _cdp_domains}

def _find_cmd(domain, method):
    d = _domains.get(domain, {})
    return first(c for c in d.get('commands', []) if c['name'] == method)

def _cmd_sig(cmd):
    _P = inspect.Parameter
    ps = [_P(p['name'], _P.KEYWORD_ONLY, default=None if p.get('optional') else _P.empty) for p in cmd.get('parameters', [])]
    return inspect.Signature([_P('sid', _P.KEYWORD_ONLY, default=None)] + ps)

def _cmd_doc(domain, method, cmd):
    doc = f"{domain}.{method}"
    if cmd.get('description'): doc += f" - {cmd['description']}"
    for p in cmd.get('parameters', []):
        opt = ' (optional)' if p.get('optional') else ''
        doc += f"\n  {p['name']}{opt}: {p.get('description', '')}"
    return doc

In [ ]:
#| export
class CDPMethod:
    def __init__(self, cdp, domain, method):
        store_attr()
        if (cmd := _find_cmd(domain, method)):
            self.__doc__ = _cmd_doc(domain, method, cmd)
            self.__signature__ = _cmd_sig(cmd)

    async def __call__(self, sid=None, **kw): return await self.cdp(f'{self.domain}.{self.method}', sid=sid, **kw)

class CDPDomain:
    def __init__(self, cdp, domain): store_attr()

    def __getattr__(self, method):
        if method.startswith('_'): raise AttributeError()
        return CDPMethod(self.cdp, self.domain, method)

    def __dir__(self):
        d = _domains.get(self.domain, {})
        return [c['name'] for c in d.get('commands', [])] + [e['name'] for e in d.get('events', [])]

In [ ]:
#| export
@patch
async def attach(self:CDP, tid):
    return await self.target.attachToTarget(targetId=tid, flatten=True)

@patch
async def eval(self:CDP, expr, sid=None):
    return (await self.runtime.evaluate(sid=sid, expression=expr)).get('value')

In [ ]:
tid = pg['targetId']
sid = await cdp.attach(tid)
await cdp.eval('document.title', sid)

'Example Domain'

In [ ]:
#| export
@patch
@asynccontextmanager
async def on(self:CDP, event):
    q = asyncio.Queue()
    self._events.setdefault(event, []).append(q)
    try: yield q
    finally: self._events[event].remove(q)

@patch
async def wait_event(self:CDP, event, timeout=10):
    async with self.on(event) as q:
        return await asyncio.wait_for(q.get(), timeout)


In [ ]:
#| export
@patch
async def wait_load(self:CDP, sid=None, timeout=10):
    while True:
        e = await self.wait_event('Page.lifecycleEvent', timeout=timeout)
        print(e)
        if e['params'].get('name') == 'networkIdle': return

@patch
async def wait_for(self:CDP, expr, sid=None, timeout=10):
    "Wait for JS expression to be truthy, return its value"
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        r = await self.eval(expr, sid)
        if r: return r
        await asyncio.sleep(0.05)
    raise TimeoutError(f'Timed out waiting for: {expr}')

@patch
async def wait_for_selector(self:CDP, sel, sid=None, timeout=10):
    "Wait for CSS selector to match an element"
    return await self.wait_for(f'!!document.querySelector("{sel}")', sid, timeout)

In [ ]:
t = await cdp.target.createTarget(url='about:blank')
sid = await cdp.attach(t)
page = await cdp.page.enable(sid=sid)

await cdp.page.navigate(sid=sid, url='https://httpbingo.org/forms/post')
await cdp.wait_for_selector('form', sid)

True

In [ ]:
await cdp.target.closeTarget(targetId=t)

True

In [ ]:
#| export
def _copy_meta(src, dest):
    if hasattr(src, '__doc__'): dest.__doc__ = src.__doc__
    if hasattr(src, '__signature__'):
        dest.__signature__ = src.__signature__.replace(
            parameters=[p for p in src.__signature__.parameters.values() if p.name != 'sid'])
    return dest

In [ ]:
#| export
class PageDomain:
    def __init__(self, sid, domain): store_attr()
    def __dir__(self): return self.domain.__dir__()

    def __getattr__(self, name):
        if name.startswith('_'): raise AttributeError(name)
        m = getattr(self.domain, name)
        async def _f(**kw): return await m(sid=self.sid, **kw)
        return _copy_meta(m, _f)

In [ ]:
#| export
class Page:
    def __init__(self, cdp, t, sid, owned=False): store_attr()

    @classmethod
    async def new(cls, t=None, cdp=None, **kwargs):
        if not cdp: cdp,owned = await CDP.connect(**kwargs),True
        else: owned = False
        if not t: t = await cdp.target.createTarget(url='about:blank')
        sid = await cdp.attach(t)
        self = cls(cdp, t, sid, owned=owned)
        await cdp.page.enable(sid=sid)
        return self

    def __getattr__(self, name):
        if name.startswith('_'): raise AttributeError(name)
        o = getattr(self.cdp, name)
        if isinstance(o, CDPDomain): return PageDomain(self.sid, o)
        if not callable(o): return o
        async def _f(*a, **kw): return await o(*a, sid=self.sid, **kw)
        return _copy_meta(o, _f)

    async def close(self):
        # Ignore errors if already closed
        try: await self.cdp.target.closeTarget(targetId=self.t)
        except RuntimeError: pass
        if self.owned: await self.cdp.close()

    @property
    def is_open(self): return self.cdp.is_open

    def __dir__(self): return self.cdp.__dir__()

In [ ]:
#| export
@patch
async def active_page(self:CDP):
    for t in (await self('Target.getTargets')):
        if t['type'] != 'page': continue
        sid = await self.attach(t['targetId'])
        if await self.eval('document.hasFocus()', sid): return t, sid
        await self.target.detachFromTarget(sessionId=sid)

@patch(cls_method=True)
async def remote_page(cls:CDP, port=9223, debug=None):
    "Connect via remote debugging and return Page for the active tab"
    cdp = await cls.remote(port=port, debug=debug)
    result = await cdp.active_page()
    if not result: raise RuntimeError("No focused page found")
    t, sid = result
    return Page(cdp, t['targetId'], sid, owned=True)


`remote_page` is the quickest start against a debug Chrome: connect and drive whichever tab is focused.

In [ ]:
page = await CDP.remote_page(port=lcdp.port)
await page.eval('document.title')

'Example Domain'

In [ ]:
#| export
@patch
async def new_page(self:CDP):
    "Create a new tab, return Page"
    t = await self.target.createTarget(url='about:blank')
    await asyncio.sleep(0.1)
    return await Page.new(t, self)

In [ ]:
page = await cdp.new_page()
await page.page.navigate(url='https://httpbingo.org/forms/post')
await page.wait_for_selector('form')

True

In [ ]:
page.is_open

True

In [ ]:
await page.close()

In [ ]:
page.is_open

True

In [ ]:
#| export
@patch
async def wait_for_ready(self:CDP, sid=None, timeout=10, idle_ms=500):
    "Wait until network is idle for idle_ms"
    await self.network.enable(sid=sid)
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        async with self.on('Network.requestWillBeSent') as q:
            try: await asyncio.wait_for(q.get(), timeout=idle_ms/1000)
            except asyncio.TimeoutError: return
    raise TimeoutError('Timed out waiting for network idle')


@patch
@asynccontextmanager
async def wait_ready(self:CDP, sid=None, timeout=10, idle_ms=500):
    "Context manager: subscribes before action, waits for load+idle after"
    await self.page.setLifecycleEventsEnabled(sid=sid, enabled=True)
    async with self.on('Page.loadEventFired') as q_load:
        yield
        await asyncio.wait_for(q_load.get(), timeout=timeout)
        await self.wait_for_ready(sid=sid, timeout=timeout, idle_ms=idle_ms)

@patch
async def goto(self:CDP, url, sid=None, **kwargs):
    "Navigate to url and wait for load+idle"
    async with self.wait_ready(sid=sid, **kwargs): await self.page.navigate(sid=sid, url=url)

In [ ]:
page = await Page.new(cdp=cdp)
await page.goto('https://httpbingo.org/forms/post')

In [ ]:
#| export
@patch
async def screenshot(self:CDP, sid=None, full=False):
    "Screenshot of the viewport, or the whole scrollable page if `full`"
    from IPython.display import Image
    b64 = await self.page.captureScreenshot(sid=sid, format='png', captureBeyondViewport=full)
    return Image(base64.b64decode(b64))

In [ ]:
await asyncio.sleep(0.2)

In [ ]:
img = await page.screenshot()
# img

In [ ]:
await page.close()

In [ ]:
await cdp.close()

## LLMs and accessibility

In [ ]:
cdp = await CDP.remote(port=lcdp.port)

In [ ]:
page = await cdp.new_page()
await page.goto('https://httpbingo.org/forms/post')
await page.eval('document.title')

''

In [ ]:
await page.accessibility.enable()
tree = await page.accessibility.getFullAXTree()
len(tree)

90

In [ ]:
tree[0]

{'nodeId': '2',
 'ignored': False,
 'role': {'type': 'internalRole', 'value': 'RootWebArea'},
 'chromeRole': {'type': 'internalRole', 'value': 144},
 'name': {'type': 'computedString',
  'value': '',
  'sources': [{'type': 'relatedElement', 'attribute': 'aria-labelledby'},
   {'type': 'attribute', 'attribute': 'aria-label'},
   {'type': 'attribute', 'attribute': 'aria-label', 'superseded': True},
   {'type': 'relatedElement', 'nativeSource': 'title'}]},
 'properties': [{'name': 'focusable',
   'value': {'type': 'booleanOrUndefined', 'value': True}},
  {'name': 'focused', 'value': {'type': 'booleanOrUndefined', 'value': True}},
  {'name': 'url',
   'value': {'type': 'string', 'value': 'https://httpbingo.org/forms/post'}}],
 'childIds': ['18'],
 'backendDOMNodeId': 2,
 'frameId': '0DAA2D1AB6D78BA6777EBEAC8A2F03F7'}

In [ ]:
await page.close()

In [ ]:
#| export
class AXNode:
    "Chrome accessibility tree node with compact repr"
    def __init__(self, raw):
        self.role = nested_idx(raw, 'role', 'value') or ''
        self.name = nested_idx(raw, 'name', 'value') or ''
        self.props = {p['name']: nested_idx(p, 'value', 'value') for p in raw.get('properties', [])}
        self.children = []
        self.node_id = raw.get('nodeId')
        self.backend_id = raw.get('backendDOMNodeId')
        self.ignored = raw.get('ignored', False)
        self._raw = raw

    def _repr_markdown_(self, depth=0):
        pre = '  ' * depth
        ps = ''.join(f' `{k}={v}`' for k,v in self.props.items() if v not in (False, None, '', 'false'))
        bid = f' [#{self.backend_id}]' if self.backend_id else ''
        line = f'{pre}- **{self.role}** "{self.name}"{ps}{bid}'
        return '\n'.join([line] + [c._repr_markdown_(depth+1) for c in self.children])

    __str__ = __repr__ = _repr_markdown_

class AXTree(AXNode):
    def _repr_markdown_(self): return f'<div class="prose">\n\n{super()._repr_markdown_(0)}\n\n</div>'

In [ ]:
#| export
def _simplify(node):
    node.children = [n for c in node.children for n in _simplify(c)]
    if node.role in ('none', 'generic', 'paragraph') and not node.name:
        if not node.children: return []
        return node.children  # splice children up
    return [node]

def build_ax_tree(nodes):
    "Build AXNode tree from flat CDP accessibility node list"
    by_id = {}
    for raw in nodes:
        n = AXNode(raw)
        by_id[n.node_id] = n
    for raw in nodes:
        parent = by_id.get(raw.get('nodeId'))
        for cid in raw.get('childIds', []):
            if (child := by_id.get(cid)): parent.children.append(child)
    if not nodes: return
    root = by_id.get(nodes[0]['nodeId'])
    if not (result := _simplify(root)): return None
    result = result[0]
    result.__class__ = AXTree
    return result

In [ ]:
#| export
@patch
async def ax_tree(self:CDP, sid=None):
    "Get accessibility tree for session"
    await self.accessibility.enable(sid=sid)
    return build_ax_tree(await self.accessibility.getFullAXTree(sid=sid))

In [ ]:
page = await cdp.new_page()
await page.goto('https://httpbingo.org/forms/post')

In [ ]:
root = await page.ax_tree()
root

<div class="prose">

- **RootWebArea** "" `focusable=True` `focused=True` `url=https://httpbingo.org/forms/post` [#2]
  - **form** "" [#5]
    - **LabelText** "" [#22]
      - **StaticText** "Customer name: " [#63]
        - **InlineTextBox** "Customer name: "
      - **textbox** "Customer name: " `focusable=True` `editable=plaintext` `settable=True` [#6]
    - **LabelText** "" [#25]
      - **StaticText** "Telephone: " [#64]
        - **InlineTextBox** "Telephone: "
      - **textbox** "Telephone: " `focusable=True` `editable=plaintext` `settable=True` [#7]
    - **LabelText** "" [#28]
      - **StaticText** "E-mail address: " [#65]
        - **InlineTextBox** "E-mail address: "
      - **textbox** "E-mail address: " `focusable=True` `editable=plaintext` `settable=True` [#8]
    - **group** "Pizza Size" [#30]
      - **Legend** "" [#31]
        - **StaticText** "Pizza Size" [#66]
          - **InlineTextBox** "Pizza Size"
      - **radio** " Small" `focusable=True` [#10]
      - **radio** " Medium" `focusable=True` [#11]
      - **radio** " Large" `focusable=True` [#12]
    - **group** "Pizza Toppings" [#38]
      - **Legend** "" [#39]
        - **StaticText** "Pizza Toppings" [#70]
          - **InlineTextBox** "Pizza Toppings"
      - **checkbox** " Bacon" `focusable=True` [#13]
      - **checkbox** " Extra Cheese" `focusable=True` [#14]
      - **checkbox** " Onion" `focusable=True` [#15]
      - **checkbox** " Mushroom" `focusable=True` [#16]
    - **LabelText** "" [#49]
      - **StaticText** "Preferred delivery time: " [#75]
        - **InlineTextBox** "Preferred delivery time: "
      - **InputTime** "Preferred delivery time: " `focusable=True` `settable=True` [#17]
        - **spinbutton** "Hours Hours" `focusable=True` `settable=True` `valuemin=1` `valuemax=12` [#53]
          - **StaticText** "--" [#76]
            - **InlineTextBox** "--"
        - **StaticText** ":" [#77]
          - **InlineTextBox** ":"
        - **spinbutton** "Minutes Minutes" `focusable=True` `settable=True` `valuemax=59` [#55]
          - **StaticText** "--" [#78]
            - **InlineTextBox** "--"
        - **StaticText** " " [#79]
          - **InlineTextBox** " "
        - **spinbutton** "AM/PM AM/PM" `focusable=True` `settable=True` `valuemin=1` `valuemax=2` [#57]
          - **StaticText** "--" [#80]
            - **InlineTextBox** "--"
        - **button** "Show time picker" `focusable=True` `hasPopup=menu` [#3]
    - **LabelText** "" [#59]
      - **StaticText** "Delivery instructions: " [#81]
        - **InlineTextBox** "Delivery instructions: "
      - **textbox** "Delivery instructions: " `focusable=True` `editable=plaintext` `settable=True` `multiline=True` [#9]
    - **button** "Submit order" `focusable=True` [#62]
      - **StaticText** "Submit order" [#82]
        - **InlineTextBox** "Submit order"

</div>

In [ ]:
#| export
@patch
def find(self:AXNode, role=None, name=None):
    "Find first descendant matching role and/or name substring"
    if (not role or role == self.role) and (not name or name in self.name): return self
    for c in self.children:
        if (r := c.find(role, name)): return r

@patch
def find_id(self:AXNode, role=None, name=None):
    "Find first descendant matching role and/or name substring"
    res = self.find(role, name)
    return res.backend_id if res else None

@patch
def find_all(self:AXNode, role=None, name=None):
    "Find all descendants matching role and/or name substring"
    res = []
    if (not role or role == self.role) and (not name or name in self.name): res.append(self)
    for c in self.children: res += c.find_all(role, name)
    return res

In [ ]:
nmid = root.find_id('textbox', 'Customer name')
phid = root.find_id('textbox', 'Telephone')
nmid

6

In [ ]:
#| export
@patch
async def js_node(self:CDP, fn, backendNodeId, sid=None):
    obj = await self.DOM.resolveNode(sid=sid, backendNodeId=backendNodeId)
    return await self.runtime.callFunctionOn(sid=sid, functionDeclaration=fn,
                                              objectId=obj['objectId'])

@patch
async def js_node_run(self:CDP, code, backendNodeId, sid=None):
    return await self.js_node(f'function() {{ {code} }}', backendNodeId, sid=sid)

@patch
async def click(self:CDP, backendNodeId, sid=None):
    await self.js_node_run('this.click()', backendNodeId, sid=sid)

In [ ]:
await page.DOM.focus(backendNodeId=nmid)
await page.input.insertText(text='Jeremy Howard')

await page.DOM.focus(backendNodeId=phid)
await page.input.insertText(text='555-1234')

{}

In [ ]:
await page.click(root.find_id('radio', 'Large'))
await page.click(root.find_id('checkbox', 'Extra Cheese'))

In [ ]:
#| export
@patch
async def fill_text(self:CDP, backendNodeId, text, sid=None):
    await self.DOM.focus(sid=sid, backendNodeId=backendNodeId)
    await self.input.insertText(sid=sid, text=text)

In [ ]:
await page.fill_text(root.find_id('textbox', 'Delivery'), 'Ring the doorbell twice')

In [ ]:
await page.js_node_run('this.value = "18:30"', root.find_id('InputTime', 'delivery time'))

{'type': 'undefined'}

In [ ]:
#| export
@patch
async def click_and_wait(self:CDP, backendNodeId, sid=None, **kwargs):
    "Click element and wait for load+idle"
    async with self.wait_ready(sid=sid, **kwargs): await self.js_node_run('this.click()', backendNodeId, sid=sid)

In [ ]:
await page.click_and_wait(root.find_id('button', 'Submit order'))

## Debugging

Helpers for debugging apps: buffered console and network history, auto-handled dialogs, and a few conveniences. CDP only delivers events once the relevant domain is enabled, so each `start_*` helper enables it and buffers from that moment on.

In [ ]:
#| export
class _EvtBuf:
    "Accumulate events from subscribed queues, draining on read"
    def __init__(self, cdp, *events):
        self.q,self.items = asyncio.Queue(),[]
        for e in events: cdp._events.setdefault(e, []).append(self.q)
    def drain(self):
        while not self.q.empty(): self.items.append(self.q.get_nowait())
        return self.items

def _fmt_console(m):
    p = m['params']
    if (d := p.get('exceptionDetails')): return f"error: {nested_idx(d, 'exception', 'description') or d['text']}"
    return f"{p['type']}: {' '.join(str(a.get('value', a.get('description', ''))) for a in p['args'])}"

@patch
async def start_console(self:CDP, sid=None):
    "Enable and start buffering console messages and uncaught exceptions"
    await self.runtime.enable(sid=sid)
    self._console = _EvtBuf(self, 'Runtime.consoleAPICalled', 'Runtime.exceptionThrown')

@patch
async def console(self:CDP, pattern=None, sid=None):
    "Console/exception messages buffered since `start_console`, filtered by regex `pattern`"
    msgs = self._console.drain()
    if sid: msgs = [m for m in msgs if m.get('sessionId') == sid]
    res = [_fmt_console(m) for m in msgs]
    return [s for s in res if re.search(pattern, s)] if pattern else res

`start_console` begins capture, and `console` returns everything seen so far; `error:` entries are uncaught exceptions, with their stack. Messages logged before `start_console` was called are never seen, so call it right after creating a page.

In [ ]:
from fastcore.test import *

In [ ]:
page = await cdp.new_page()
await page.start_console()
await page.eval(r'console.log("hello", 42); console.warn("watch out")')
await page.eval(r'setTimeout(() => { throw new Error("boom") }, 0)')
await asyncio.sleep(0.1)
await page.console()

['log: hello 42',
 'warning: watch out',
 'error: Error: boom\n    at <anonymous>:1:26']

The `pattern` regex filters entries:

In [ ]:
test_eq(await page.console(r'watch'), ['warning: watch out'])
assert any('boom' in s for s in await page.console(r'error:'))

In [ ]:
#| export
@patch
async def start_network(self:CDP, sid=None):
    "Enable and start buffering network responses"
    await self.network.enable(sid=sid)
    self._network = _EvtBuf(self, 'Network.responseReceived')

@patch
async def requests(self:CDP, pattern=None, sid=None):
    "(status,url,requestId) of responses buffered since `start_network`, url filtered by regex `pattern`"
    msgs = self._network.drain()
    if sid: msgs = [m for m in msgs if m.get('sessionId') == sid]
    res = [(p['response']['status'], p['response']['url'], p['requestId']) for m in msgs for p in [m['params']]]
    return [r for r in res if re.search(pattern, r[1])] if pattern else res

@patch
async def response_body(self:CDP, requestId, sid=None):
    "Body of a response seen by `start_network`, decoded if base64"
    r = await self.network.getResponseBody(sid=sid, requestId=requestId)
    return base64.b64decode(r['body']).decode() if r['base64Encoded'] else r['body']

`requests` answers "what did the page load, and with what status?", and `response_body` fetches a body by the returned request id (Chrome only keeps bodies while the page is alive).

In [ ]:
await page.start_network()
await page.goto('https://httpbingo.org/forms/post')
await page.requests(r'httpbingo')

[(200, 'https://httpbingo.org/forms/post', 'D537A2E8B047C71C952C3E125CF226E1')]

In [ ]:
st,url,rid = first(await page.requests(r'forms/post'))
test_eq(st, 200)
assert '<form' in (await page.response_body(rid))

In [ ]:
#| export
@patch
async def handle_dialogs(self:CDP, accept=True, text=None, sid=None):
    "Auto-respond to JS dialogs from now on, recording (type,message) in `dialogs`"
    await self.page.enable(sid=sid)
    self.dialogs = []
    q = asyncio.Queue()
    self._events.setdefault('Page.javascriptDialogOpening', []).append(q)
    async def _h():
        while True:
            m = await q.get()
            p = m['params']
            self.dialogs.append((p['type'], p['message']))
            kw = dict(accept=accept)
            if text is not None: kw['promptText'] = text
            await self.page.handleJavaScriptDialog(sid=m.get('sessionId'), **kw)
    self._dialog_task = asyncio.create_task(_h())

Without a handler, an unexpected `alert` or `confirm` blocks the page -- and any `eval` that triggered it -- forever. After `handle_dialogs`, dialogs are answered as they open: `accept=False` dismisses them, and `text` fills prompts.

In [ ]:
await page.handle_dialogs()
ok = await page.eval(r'confirm("Proceed?")')
ok, page.dialogs

(True, [('confirm', 'Proceed?')])

In [ ]:
test_eq(ok, True)
test_eq(page.dialogs, [('confirm', 'Proceed?')])

In [ ]:
#| export
@patch
async def select_option(self:CDP, backendNodeId, value, sid=None):
    "Set a `<select>` element's value and fire its change event"
    await self.js_node_run(f'this.value = {json.dumps(value)}; this.dispatchEvent(new Event("change", {{bubbles:true}}))', backendNodeId, sid=sid)

@patch
async def wait_for_text(self:CDP, text, present=True, sid=None, timeout=10):
    "Wait for `text` to appear in (with `present=False`, disappear from) the page body"
    expr = f'document.body.innerText.includes({json.dumps(text)})'
    return await self.wait_for(expr if present else f'!({expr})', sid, timeout)

`click` on a `<select>` doesn't open native dropdowns under CDP, so `select_option` sets the value directly (firing `change` so the app reacts). `wait_for_text` complements `wait_for_selector` when the interesting change is text, e.g. htmx swaps. And `screenshot` above takes `full=True` for the whole scrollable page.

In [ ]:
await page.goto('data:text/html,<select><option>small<option>large</select><div style="height:3000px"></div>')
await page.select_option((await page.ax_tree()).find_id('combobox'), 'large')
test_eq(await page.eval(r'document.querySelector("select").value'), 'large')

In [ ]:
hs = [int.from_bytes(i.data[20:24], 'big') for i in (await page.screenshot(), await page.screenshot(full=True))]
assert hs[1] > hs[0]
hs

[1026, 6070]

In [ ]:
await page.eval(r'setTimeout(() => document.body.append(" loaded!"), 300)')
await page.wait_for_text('loaded!')
await page.eval(r'setTimeout(() => { document.body.innerHTML = "gone" }, 200)')
await page.wait_for_text('loaded!', present=False)

True

In [ ]:
await page.close()

In [ ]:
await cdp.close()

To finish, exercise the browser we launched at the start end to end, then `quit` it:

In [ ]:
page2 = await lcdp.new_page()
await page2.goto('https://example.com')
test_eq(await page2.eval('document.title'), 'Example Domain')

In [ ]:
await lcdp.quit()
assert not lcdp.is_open and lcdp.proc.returncode is not None

In [ ]:
#| export
def cdp_yolo():
    "Allow all CDP classes in safepyrun"
    from pyskills import allow
    allow({CDP:..., CDPDomain:..., CDPMethod:...})
    allow({PageDomain:..., Page:..., AXNode:..., AXTree:...})